# Qwen3-14B QLoRA SFT


In [ ]:
import os, subprocess, sys
from pathlib import Path

# Kaggle Secrets are accessed through its API, not injected into os.environ.
# Preserve explicit environment values so local/Kaggle overrides still win.
_SECRET_NAMES = (
    "HF_TOKEN",
    "CRASHDIAG_SANDBOX_URL",
    "CRASHDIAG_API_TOKEN",
    "CRASHDIAG_DATASET_RUN_ID",
    "CRASHDIAG_BASE_QWEN3_14B_RUN_ID",
    "CRASHDIAG_SFT_RUN_ID",
    "CRASHDIAG_SFT_EVAL_RUN_ID",
    "CRASHDIAG_GRPO_RUN_ID",
)
try:
    from kaggle_secrets import UserSecretsClient
except ImportError:
    pass
else:
    _secrets = UserSecretsClient()
    for _secret_name in _SECRET_NAMES:
        if os.environ.get(_secret_name):
            continue
        try:
            _secret_value = _secrets.get_secret(_secret_name)
        except Exception:
            continue
        if _secret_value:
            os.environ[_secret_name] = _secret_value

REPO_URL = os.environ.get("CRASHDIAG_REPO_URL", "https://github.com/Indium-AI-Labs/CrashDiag.git")
SOURCE_COMMIT = os.environ.get("CRASHDIAG_SOURCE_COMMIT", "main")
WORKDIR = Path("/kaggle/working/CrashDiag")
if WORKDIR.exists():
    subprocess.run(["rm", "-rf", str(WORKDIR)], check=True)
subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
subprocess.run(["git", "-C", str(WORKDIR), "checkout", SOURCE_COMMIT], check=True)
os.chdir(WORKDIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "bitsandbytes"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], check=True)
print("checked_out_source_commit=" + subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
import os

BASE_MODEL = "Qwen/Qwen3-14B"
MODEL_SLUG = "qwen3_14b"
BUCKET_ID = "devaanshpa/CrashDiag"
DATASET_RUN_ID = os.environ.get("CRASHDIAG_DATASET_RUN_ID", "").strip()
def ist_run_id(stage):
    return datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%Y%m%dT%H%M%SIST") + f"-{MODEL_SLUG}-{stage}"
if not DATASET_RUN_ID:
    raise RuntimeError("Set CRASHDIAG_DATASET_RUN_ID to the fresh dataset-generation run ID.")
print(f"base_model={BASE_MODEL}")
print(f"dataset_run_id={DATASET_RUN_ID}")


In [ ]:
from pathlib import Path
from training.artifacts import ArtifactConfig, ArtifactUploader

SFT_RUN_ID = os.environ.get("CRASHDIAG_SFT_RUN_ID") or ist_run_id("sft")
DATASET_DIR = Path("artifacts/datasets")
ArtifactUploader(ArtifactConfig(bucket_id=BUCKET_ID, run_id=DATASET_RUN_ID, token=os.environ["HF_TOKEN"])).download_stage("datasets", DATASET_DIR)
print(f"SFT_RUN_ID={SFT_RUN_ID}")


In [ ]:
import subprocess, sys

command = [
    sys.executable, "-m", "accelerate.commands.launch", "--num_processes", "2",
    "-m", "training.sft",
    "--model", BASE_MODEL,
    "--dataset", str(DATASET_DIR / "sft_train.jsonl"),
    "--eval-dataset", str(DATASET_DIR / "sft_eval.jsonl"),
    "--output-dir", "outputs/sft",
    "--epochs", "2",
    "--batch-size", "1",
    "--eval-batch-size", "1",
    "--gradient-accumulation-steps", "8",
    "--learning-rate", "2e-4",
    "--load-in-4bit",
    "--precision", "fp16",
    "--report-to", "none",
    "--artifact-bucket", BUCKET_ID,
    "--run-id", SFT_RUN_ID,
]
subprocess.run(command, check=True)


In [ ]:
from IPython.display import SVG, display

REPORTS_DIR = Path("outputs/sft") / "reports"
charts = sorted(REPORTS_DIR.glob("*.svg"))
if not charts:
    raise RuntimeError(f"No SFT SVG charts were generated in {REPORTS_DIR}")
print(f"Uploaded SFT reports: hf://buckets/{BUCKET_ID}/runs/{SFT_RUN_ID}/sft/reports")
for chart in charts:
    display(SVG(filename=str(chart)))
